## Prepare Model For Inferencing

In [28]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import joblib
import pickle
import warnings
import os

In [29]:
# Suppress scikit-learn warnings about the dataset
warnings.filterwarnings("ignore", category=FutureWarning)

In [30]:
os.makedirs('server', exist_ok=True)

In [31]:
# Load the Boston Housing dataset from an online repository
# Scikit-learn has deprecated this dataset due to ethical concerns.
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
raw_df.head()

,0,1,2,3,4,5,6,7,8,9,10
0,0.00632,18.00,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3
1,396.90000,4.98,24.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.02731,0.00,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8
3,396.90000,9.14,21.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.02729,0.00,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8


In [32]:
# The dataset is structured in a way that every two rows correspond to one data point, with the first row containing the features and the second row containing the target variable (MEDV). We need to reshape the data accordingly.
# You get a new array where each row is made by joining:
# The full row from every even index,
# With the first two columns from the next (odd) row.
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

In [33]:
# Test the loaded data
print(f"Data shape: {data.shape}")
print(f"Target shape: {data.shape}")
print(f"Target range: {data.min()} to {data.max()}")

Data shape: (506, 13)
Target shape: (506, 13)
Target range: 0.0 to 711.0


In [34]:
# Define features and target (optional, but good practice)
# The actual column names are: CRIM ZN INDUS CHAS NOX RM AGE DIS RAD TAX PTRATIO B LSTAT MEDV
# We will use the target variable MEDV (median value) as the prediction target.
raw_df.head()

,0,1,2,3,4,5,6,7,8,9,10
0,0.00632,18.00,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3
1,396.90000,4.98,24.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.02731,0.00,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8
3,396.90000,9.14,21.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.02729,0.00,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8


In [35]:
raw_df.head()
raw_df.info()
raw_df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1012 entries, 0 to 1011
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       1012 non-null   float64
 1   1       1012 non-null   float64
 2   2       1012 non-null   float64
 3   3       506 non-null    float64
 4   4       506 non-null    float64
 5   5       506 non-null    float64
 6   6       506 non-null    float64
 7   7       506 non-null    float64
 8   8       506 non-null    float64
 9   9       506 non-null    float64
 10  10      506 non-null    float64
dtypes: float64(11)
memory usage: 87.1 KB


,0,1,2,3,4,5,6,7,8,9,10
count,1012.000000,1012.000000,1012.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000,506.000000
mean,180.143778,12.008350,16.834792,0.069170,0.554695,6.284634,68.574901,3.795043,9.549407,408.237154,18.455534
std,188.132839,17.250728,9.912616,0.253994,0.115878,0.702617,28.148861,2.105710,8.707259,168.537116,2.164946
min,0.006320,0.000000,0.460000,0.000000,0.385000,3.561000,2.900000,1.129600,1.000000,187.000000,12.600000
25%,0.257830,0.000000,8.375000,0.000000,0.449000,5.885500,45.025000,2.100175,4.000000,279.000000,17.400000
50%,24.021000,7.240000,18.100000,0.000000,0.538000,6.208500,77.500000,3.207450,5.000000,330.000000,19.050000
75%,391.435000,16.780000,21.890000,0.000000,0.624000,6.623500,94.075000,5.188425,24.000000,666.000000,20.200000
max,396.900000,100.000000,50.000000,1.000000,0.871000,8.780000,100.000000,12.126500,24.000000,711.000000,22.000000


In [36]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)

In [37]:
# scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # ✅ Scale test set with training parameters

In [38]:
X_train_scaled.shape

(404, 13)

In [39]:
# Initialize and train a model (e.g., RandomForestRegressor)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

RandomForestRegressor(random_state=42)

In [40]:
# Evaluate the model (optional)
predictions = model.predict(X_test_scaled)
print(f"Mean Squared Error: {mean_squared_error(y_test, predictions)}")

Mean Squared Error: 7.912745333333333


In [41]:
print(f"R2 Score: {r2_score(y_test, predictions)}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, predictions)}")

R2 Score: 0.8920995891343227
Mean Absolute Error: 2.041078431372549


In [42]:
# Save the trained model as a joblib file
# Save both the model and the scaler
save_path = 'server/boston_housing_prediction.joblib'
joblib.dump({'model': model, 'scaler': scaler}, save_path)
print("Model successfully saved as boston_housing_prediction.joblib")

Model successfully saved as boston_housing_prediction.joblib


In [43]:
# Save the trained model as a pickle file
with open("server/boston_housing_prediction.pkl", "wb") as f:
    pickle.dump(model, f)

In [44]:
print("Models saved in server folder")

Models saved in server folder
